## Building the price loader

Goal: for every entity that was ever a member of the S&P 500 (per `universe_spans`), retrieve
daily OHLCV prices, using the historically correct ticker for each date range (per
`ticker_history`), cached locally rather than fetched live.

Start small: pull one familiar, currently listed ticker with no date restriction, and look at
the raw shape `yfinance` returns, before designing anything around the harder cases (renames,
delistings, batch fetches).


## Part 1: the vendor's raw shape

Before designing anything, look at what `yfinance` actually returns for one familiar, currently
listed ticker, full history, no date restriction.

In [1]:
import yfinance as yf

aapl = yf.Ticker("AAPL")
history = aapl.history(period="max")

print(history.shape)
print(history.index.min(), "to", history.index.max())
print(list(history.columns))
history.head()

(11500, 7)
1980-12-12 00:00:00-05:00 to 2026-07-31 00:00:00-04:00
['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
1980-12-12 00:00:00-05:00,0.098207,0.098634,0.098207,0.098207,469033600,0.0,0.0
1980-12-15 00:00:00-05:00,0.093510,0.093510,0.093083,0.093083,175884800,0.0,0.0
1980-12-16 00:00:00-05:00,0.086678,0.086678,0.086251,0.086251,105728000,0.0,0.0
1980-12-17 00:00:00-05:00,0.088386,0.088813,0.088386,0.088386,86441600,0.0,0.0
1980-12-18 00:00:00-05:00,0.090949,0.091376,0.090949,0.090949,73449600,0.0,0.0


## Part 2: renames, recycling, and the vendor's folding behavior

Two things stand out from Part 1 worth chasing: no `Adj Close` column (adjustment is baked
into OHLC directly), and a tz-aware index. Neither turns out to be the hard part. The hard
part, tested here: what happens when a ticker has been renamed, reused, or retired.

In [2]:
for ticker in ["PCLN", "FRC"]:
    data = yf.Ticker(ticker).history(period="max")
    print(ticker, data.shape)
    if not data.empty:
        print(" ", data.index.min(), "to", data.index.max())


PCLN (198, 7)
  2025-10-16 00:00:00-04:00 to 2026-07-31 00:00:00-04:00


$FRC: possibly delisted; no timezone found


FRC (0, 6)


In [3]:
bkng = yf.Ticker("BKNG").history(period="max")
print("BKNG", bkng.shape)
print(" ", bkng.index.min(), "to", bkng.index.max())

frc = yf.Ticker("FRC").history(period="max")
print("FRC columns:", list(frc.columns))


$FRC: possibly delisted; no timezone found


BKNG (6876, 7)
  1999-03-31 00:00:00-05:00 to 2026-07-31 00:00:00-04:00
FRC columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']


In [4]:
gm = yf.Ticker("GM").history(period="max")
print("GM", gm.shape)
print(" ", gm.index.min(), "to", gm.index.max())


GM (3947, 7)
  2010-11-18 00:00:00-05:00 to 2026-07-31 00:00:00-04:00


## Part 3: batching mechanics

Before scaling to hundreds of tickers, check how `yf.download()` (the multi-ticker call)
behaves differently from `Ticker().history()`, and what happens when one ticker in a batch is
invalid.

In [5]:
tickers = ["AAPL", "MSFT", "PCLN", "NOTAREALTICKER"]
batch = yf.download(tickers, period="1mo")
print(batch.shape)
print(batch.columns)


[**********************75%***********            ]  3 of 4 completed$NOTAREALTICKER: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  4 of 4 completed

1 Failed download:
['NOTAREALTICKER']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


(22, 21)
MultiIndex([('Adj Close', 'NOTAREALTICKER'),
            (    'Close',           'AAPL'),
            (    'Close',           'MSFT'),
            (    'Close', 'NOTAREALTICKER'),
            (    'Close',           'PCLN'),
            (     'High',           'AAPL'),
            (     'High',           'MSFT'),
            (     'High', 'NOTAREALTICKER'),
            (     'High',           'PCLN'),
            (      'Low',           'AAPL'),
            (      'Low',           'MSFT'),
            (      'Low', 'NOTAREALTICKER'),
            (      'Low',           'PCLN'),
            (     'Open',           'AAPL'),
            (     'Open',           'MSFT'),
            (     'Open', 'NOTAREALTICKER'),
            (     'Open',           'PCLN'),
            (   'Volume',           'AAPL'),
            (   'Volume',           'MSFT'),
            (   'Volume', 'NOTAREALTICKER'),
            (   'Volume',           'PCLN')],
           names=['Price', 'Ticker'])


In [6]:
print(batch["Close"]["PCLN"].dropna())


Date
2026-07-01    32.558998
2026-07-02    31.818001
2026-07-06    32.264000
2026-07-07    31.409000
2026-07-08    31.496000
2026-07-09    31.985001
2026-07-10    31.996000
2026-07-13    31.325001
2026-07-14    31.679001
2026-07-15    31.444000
2026-07-16    30.971001
2026-07-17    30.516001
2026-07-20    30.343000
2026-07-21    31.000999
2026-07-22    31.044001
2026-07-23    30.924999
2026-07-24    30.601000
2026-07-27    30.520000
2026-07-28    30.007999
2026-07-29    29.157000
2026-07-30    30.243999
2026-07-31    30.438000
Name: PCLN, dtype: float64


## Part 4: ticker format translation

Connecting back to the real universe (`membership_on`) surfaces the first real ticker format
problem, a period-formatted multi class ticker `yfinance` doesn't recognize. Scanning the
whole universe for this pattern, then building and checking the translation.

In [7]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

import time
from src.universe.point_in_time import build_universe, membership_on

universe_spans, ticker_history = build_universe()
today_tickers = list(membership_on(universe_spans, "2026-07-29"))[:50]

start = time.time()
sample = yf.download(today_tickers, period="1mo", progress=False)
elapsed = time.time() - start

print(len(today_tickers), "tickers,", round(elapsed, 1), "seconds")
print(sample.shape)


/Users/hongxianli/Documents/data_science/capm-portfolio
50 tickers, 1.1 seconds
(22, 250)


In [8]:
bf_b = yf.Ticker("BF-B").history(period="1mo")
print(bf_b.shape)


(22, 7)


In [9]:
all_tickers = universe_spans["ticker"].unique()
dotted = sorted(t for t in all_tickers if "." in str(t))
print(len(dotted), "of", len(all_tickers))
print(dotted)


14 of 1407
['AFS.A-200011', 'AZA.A-200106', 'BF.B', 'BRK.B', 'CIT.A-200106', 'COC.B-200208', 'FTL.A-200004', 'GFS.A-199810', 'LDW.B-200012', 'NWS.A', 'RDS.A', 'TMC.A-200006', 'UA.C', 'VIA.B']


In [10]:
from src.universe.point_in_time import base_ticker

def to_yfinance_ticker(ticker):
    return base_ticker(ticker).replace(".", "-")

for t in dotted:
    print(t, "->", to_yfinance_ticker(t))


AFS.A-200011 -> AFS-A
AZA.A-200106 -> AZA-A
BF.B -> BF-B
BRK.B -> BRK-B
CIT.A-200106 -> CIT-A
COC.B-200208 -> COC-B
FTL.A-200004 -> FTL-A
GFS.A-199810 -> GFS-A
LDW.B-200012 -> LDW-B
NWS.A -> NWS-A
RDS.A -> RDS-A
TMC.A-200006 -> TMC-A
UA.C -> UA-C
VIA.B -> VIA-B


## Part 5: confirming the fetch design

Two questions worth answering with real data before writing the fetch loop: does requesting a
ticker's specific historical date range (rather than its full history) defuse the recycling
risk found in Part 2? And does the `BASE-YYYYMM` suffix stripped in Part 4 carry any date
information the fetch actually needs?

In [11]:
pcln_test = yf.Ticker("PCLN").history(start="2014-05-31", end="2018-03-31")
print(pcln_test.shape)
if not pcln_test.empty:
    print(pcln_test.index.min(), "to", pcln_test.index.max())


$PCLN: possibly delisted; no price data found  (1d 2014-05-31 -> 2018-03-31) (Yahoo error = "Data doesn't exist for startDate = 1401508800, endDate = 1522468800")


(0, 6)


In [12]:
universe_spans[universe_spans["ticker"] == "AFS.A-200011"]

,ticker,cik,start_date,end_date,source,left_censored
172,AFS.A-200011,<NA>,1998-04-08,2000-11-29,clenow_norgate,False


## Part 6: fetching one CIK's full history, a bug found by building it

Priceline/Booking Holdings (CIK 1075531) is the clean test case: exactly two `ticker_history`
segments, a real rename. The first, naive fetch-and-concatenate loop hides a serious bug that
only shows up once it's actually run.

In [13]:
ticker_history[ticker_history["cik"] == 1075531]


,cik,ticker,start_date,end_date,verified,source,original_ticker,evidence
1377,1075531,PCLN,2014-05-31,2018-03-31,True,wikipedia_revision,NaN,NaN
1378,1075531,BKNG,2018-03-31,NaN,True,wikipedia_revision,NaN,NaN


In [14]:
from pandas import pandas as pd
def fetch_cik_history(cik):
    segments = ticker_history[ticker_history["cik"] == cik].sort_values("start_date")
    frames = []
    for _, seg in segments.iterrows():
        yf_ticker = to_yfinance_ticker(seg["ticker"])
        end = seg["end_date"] if pd.notna(seg["end_date"]) else None
        data = yf.Ticker(yf_ticker).history(start=seg["start_date"], end=end)
        frames.append(data)
    return pd.concat(frames)

pcln_bkng = fetch_cik_history(1075531)
print(pcln_bkng.shape)
print(pcln_bkng.index.min(), "to", pcln_bkng.index.max())
pcln_bkng.loc["2018-03-25":"2018-04-05"]


$PCLN: possibly delisted; no price data found  (1d 2014-05-31 -> 2018-03-31) (Yahoo error = "Data doesn't exist for startDate = 1401508800, endDate = 1522468800")


(2095, 8)
2018-04-02 00:00:00-04:00 to 2026-07-31 00:00:00-04:00


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2018-04-02 00:00:00-04:00,81.094580,81.234805,78.724522,79.544701,NaN,9307500.0,0.0,0.0
2018-04-03 00:00:00-04:00,80.294381,81.469029,79.748773,80.982559,NaN,9460000.0,0.0,0.0
2018-04-04 00:00:00-04:00,80.028432,81.543057,78.891384,81.117302,NaN,9637500.0,0.0,0.0
2018-04-05 00:00:00-04:00,81.372664,82.349122,81.116897,81.903786,NaN,8002500.0,0.0,0.0


In [15]:
def fetch_cik_history(cik):
    segments = ticker_history[ticker_history["cik"] == cik].sort_values("start_date")
    still_active = pd.isna(segments.iloc[-1]["end_date"])

    if still_active:
        current_ticker = to_yfinance_ticker(segments.iloc[-1]["ticker"])
        earliest_start = segments.iloc[0]["start_date"]
        return yf.Ticker(current_ticker).history(start=earliest_start)

    frames = []
    for _, seg in segments.iterrows():
        yf_ticker = to_yfinance_ticker(seg["ticker"])
        frames.append(yf.Ticker(yf_ticker).history(start=seg["start_date"], end=seg["end_date"]))
    return pd.concat(frames)

pcln_bkng = fetch_cik_history(1075531)
print(pcln_bkng.shape)
print(pcln_bkng.index.min(), "to", pcln_bkng.index.max())


(3060, 7)
2014-06-02 00:00:00-04:00 to 2026-07-31 00:00:00-04:00


## Part 7: dual class shares

Searching for a genuine multi-rename CIK (to check whether the "fetch by current ticker"
fix generalizes past one rename) instead surfaces something bigger: a company with two
simultaneously traded share classes, which the current design would silently drop one of.

In [16]:
segment_counts = ticker_history.groupby("cik").size()
multi_rename = segment_counts[segment_counts >= 3].sort_values(ascending=False)
print(len(multi_rename))
multi_rename.head(10)


35


cik
1564708    260
1652044    243
1437107    185
1754301    141
1336917    139
1308161    120
1288776     49
1166691      8
798354       4
896159       4
dtype: int64

In [17]:
ticker_history[ticker_history["cik"] == 1564708].sort_values("start_date")


,cik,ticker,start_date,end_date,verified,source,original_ticker,evidence
2080,1564708,NWSA,2014-05-31,2015-09-30,True,wikipedia_revision,NaN,NaN
2081,1564708,NWS,2015-09-30,2015-10-31,True,wikipedia_revision,NaN,NaN
2082,1564708,NWSA,2015-10-31,2015-10-31,True,wikipedia_revision,NaN,NaN
2083,1564708,NWS,2015-10-31,2015-11-30,True,wikipedia_revision,NaN,NaN
2084,1564708,NWSA,2015-11-30,2015-11-30,True,wikipedia_revision,NaN,NaN
...,...,...,...,...,...,...,...,...
2335,1564708,NWS,2026-04-30,2026-05-31,True,wikipedia_revision,NaN,NaN
2336,1564708,NWSA,2026-05-31,2026-05-31,True,wikipedia_revision,NaN,NaN
2337,1564708,NWS,2026-05-31,2026-06-30,True,wikipedia_revision,NaN,NaN
2338,1564708,NWSA,2026-06-30,2026-06-30,True,wikipedia_revision,NaN,NaN


In [18]:
def ticker_spans_for_cik(cik):
    rows = ticker_history[ticker_history["cik"] == cik]
    spans = []
    for ticker, group in rows.groupby("ticker"):
        start = group["start_date"].min()
        still_current = group["end_date"].isna().any()
        end = None if still_current else group["end_date"].max()
        spans.append({"ticker": ticker, "start_date": start, "end_date": end})
    return pd.DataFrame(spans)

print(ticker_spans_for_cik(1564708))   # News Corp: expect NWSA and NWS as two separate rows
print(ticker_spans_for_cik(1075531))   # Priceline/Booking: must still give exactly PCLN, BKNG


  ticker  start_date    end_date
0    NWS  2015-09-30         NaN
1   NWSA  2014-05-31  2026-06-30
  ticker  start_date    end_date
0   BKNG  2018-03-31         NaN
1   PCLN  2014-05-31  2018-03-31


In [19]:
nwsa = yf.Ticker("NWSA").history(start="2014-05-31")
print(nwsa.shape)
print(nwsa.index.min(), "to", nwsa.index.max())


(3060, 7)
2014-06-02 00:00:00-04:00 to 2026-07-31 00:00:00-04:00


In [20]:
def ticker_spans_for_cik(cik):
    rows = ticker_history[ticker_history["cik"] == cik]
    cik_start = rows["start_date"].min()
    cik_active = rows["end_date"].isna().any()

    spans = []
    for ticker, group in rows.groupby("ticker"):
        fragmented = len(group) > 1
        still_current = group["end_date"].isna().any() or (fragmented and cik_active)
        start = cik_start if still_current else group["start_date"].min()
        end = None if still_current else group["end_date"].max()
        spans.append({"ticker": ticker, "start_date": start, "end_date": end})
    return pd.DataFrame(spans)

print(ticker_spans_for_cik(1564708))
print(ticker_spans_for_cik(1075531))


  ticker  start_date end_date
0    NWS  2014-05-31     None
1   NWSA  2014-05-31     None
  ticker  start_date    end_date
0   BKNG  2014-05-31         NaN
1   PCLN  2014-05-31  2018-03-31


In [21]:
def fetch_cik_prices(cik):
    spans = ticker_spans_for_cik(cik)
    results = {}
    for _, row in spans.iterrows():
        yf_ticker = to_yfinance_ticker(row["ticker"])
        end = None if pd.isna(row["end_date"]) else row["end_date"]
        data = yf.Ticker(yf_ticker).history(start=row["start_date"], end=end)
        results[row["ticker"]] = data
    return results

newscorp = fetch_cik_prices(1564708)
for ticker, data in newscorp.items():
    print(ticker, data.shape, data.index.min(), "to", data.index.max())


NWS (3060, 7) 2014-06-02 00:00:00-04:00 to 2026-07-31 00:00:00-04:00
NWSA (3060, 7) 2014-06-02 00:00:00-04:00 to 2026-07-31 00:00:00-04:00


In [22]:
for cik in [1288776, 1166691, 798354, 896159]:
    print(cik)
    print(ticker_history[ticker_history["cik"] == cik][["ticker", "start_date", "end_date"]])


1288776
     ticker  start_date    end_date
1480   GOOG  2014-05-31  2014-05-31
1481  GOOGL  2014-05-31  2014-06-30
1482   GOOG  2014-06-30  2014-06-30
1483  GOOGL  2014-06-30  2014-07-31
1484   GOOG  2014-07-31  2014-07-31
1485  GOOGL  2014-07-31  2014-08-31
1486   GOOG  2014-08-31  2014-08-31
1487  GOOGL  2014-08-31  2014-09-30
1488   GOOG  2014-09-30  2014-10-31
1489  GOOGL  2014-10-31  2014-10-31
1490   GOOG  2014-10-31  2014-11-30
1491  GOOGL  2014-11-30  2014-11-30
1492   GOOG  2014-11-30  2014-12-31
1493  GOOGL  2014-12-31  2014-12-31
1494   GOOG  2014-12-31  2015-01-31
1495  GOOGL  2015-01-31  2015-01-31
1496   GOOG  2015-01-31  2015-02-28
1497  GOOGL  2015-02-28  2015-02-28
1498   GOOG  2015-02-28  2015-03-31
1499  GOOGL  2015-03-31  2015-03-31
1500   GOOG  2015-03-31  2015-04-30
1501  GOOGL  2015-04-30  2015-04-30
1502   GOOG  2015-04-30  2015-05-31
1503  GOOGL  2015-05-31  2015-05-31
1504   GOOG  2015-05-31  2015-06-30
1505  GOOGL  2015-06-30  2015-06-30
1506   GOOG  2015-06

In [23]:
googl = yf.Ticker("GOOGL").history(start="2014-05-31")
print(googl.shape)
print(googl.index.min(), "to", googl.index.max())


(3060, 7)
2014-06-02 00:00:00-04:00 to 2026-07-31 00:00:00-04:00


In [24]:
cmcsk = yf.Ticker("CMCSK").history(period="max")
print(cmcsk.shape)
if not cmcsk.empty:
    print(cmcsk.index.min(), "to", cmcsk.index.max())


$CMCSK: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)


(0, 6)


In [25]:
def classify_ticker(ticker, expected_start, tolerance_days=45):
    yf_ticker = to_yfinance_ticker(ticker)
    probe = yf.Ticker(yf_ticker).history(period="max")
    if probe.empty:
        return "retired"
    actual_start = probe.index.min().tz_localize(None).normalize()
    if actual_start <= pd.Timestamp(expected_start) + pd.Timedelta(days=tolerance_days):
        return "current"
    return "recycled"

tests = [
    ("PCLN", "2014-05-31"),
    ("CMCSK", "2015-09-30"),
    ("GOOGL", "2014-05-31"),
    ("NWSA", "2014-05-31"),
    ("BKNG", "2014-05-31"),
]
for ticker, expected in tests:
    print(ticker, "->", classify_ticker(ticker, expected))


$CMCSK: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)


PCLN -> recycled
CMCSK -> retired
GOOGL -> current
NWSA -> current
BKNG -> current


In [26]:
def ticker_spans_for_cik(cik):
    rows = ticker_history[ticker_history["cik"] == cik]
    cik_start = rows["start_date"].min()

    spans = []
    for ticker, group in rows.groupby("ticker"):
        own_start = group["start_date"].min()
        own_end = None if group["end_date"].isna().any() else group["end_date"].max()
        status = classify_ticker(ticker, own_start)
        if status == "current":
            start, end = cik_start, None
        else:
            start, end = own_start, own_end
        spans.append({"ticker": ticker, "start_date": start, "end_date": end, "status": status})
    return pd.DataFrame(spans)

for cik in [1075531, 1564708, 1288776, 1166691]:
    print(cik)
    print(ticker_spans_for_cik(cik))


1075531
  ticker  start_date    end_date    status
0   BKNG  2014-05-31         NaN   current
1   PCLN  2014-05-31  2018-03-31  recycled
1564708
  ticker  start_date end_date   status
0    NWS  2014-05-31     None  current
1   NWSA  2014-05-31     None  current
1288776
  ticker  start_date end_date   status
0   GOOG  2014-05-31     None  current
1  GOOGL  2014-05-31     None  current
1166691


$CMCSK: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)


  ticker  start_date    end_date   status
0  CMCSA  1996-01-02         NaN  current
1  CMCSK  2015-09-30  2015-12-31  retired


In [27]:
ticker_spans_for_cik(798354)


$FI: possibly delisted; no timezone found


,ticker,start_date,end_date,status
0,FI,2023-06-30,2025-11-30,retired
1,FISV,2001-04-02,NaN,current


In [28]:
fisv = yf.Ticker("FISV").history(start="2001-04-02")
print(fisv.loc["2024-01-01":"2024-01-10"])


                                 Open        High         Low       Close  \
Date                                                                        
2024-01-02 00:00:00-05:00  132.330002  133.669998  131.940002  133.080002   
2024-01-03 00:00:00-05:00  133.009995  133.009995  131.410004  131.750000   
2024-01-04 00:00:00-05:00  132.160004  133.279999  131.949997  133.000000   
2024-01-05 00:00:00-05:00  133.210007  133.699997  132.149994  132.570007   
2024-01-08 00:00:00-05:00  134.300003  135.270004  133.669998  135.229996   
2024-01-09 00:00:00-05:00  134.339996  135.279999  134.339996  135.100006   
2024-01-10 00:00:00-05:00  136.000000  136.479996  134.940002  135.399994   

                            Volume  Dividends  Stock Splits  
Date                                                         
2024-01-02 00:00:00-05:00  3227800        0.0           0.0  
2024-01-03 00:00:00-05:00  3710700        0.0           0.0  
2024-01-04 00:00:00-05:00  2839200        0.0           0.

In [29]:
uaa_cik = ticker_history[ticker_history["ticker"].isin(["UAA", "UA", "UA.C"])]["cik"].unique()
print("Under Armour CIK(s):", uaa_cik)

for cik in [1652044, 1437107, 1754301, 1336917, 1308161]:
    print(cik)
    print(ticker_history[ticker_history["cik"] == cik]["ticker"].value_counts())


Under Armour CIK(s): <IntegerArray>
[1336917]
Length: 1, dtype: Int64
1652044
ticker
GOOGL    122
GOOG     121
Name: count, dtype: int64
1437107
ticker
DISCA    92
DISCK    92
WBD       1
Name: count, dtype: int64
1754301
ticker
FOXA    71
FOX     70
Name: count, dtype: int64
1336917
ticker
UA      70
UAA     65
UA.C     4
Name: count, dtype: int64
1308161
ticker
FOXA    60
FOX     60
Name: count, dtype: int64


In [30]:
googl_all = yf.Ticker("GOOGL").history(period="max")
print(googl_all.loc["2016-05-25":"2016-06-05"])


                                Open       High        Low      Close  \
Date                                                                    
2016-05-25 00:00:00-04:00  36.426995  36.669345  36.308051  36.580631   
2016-05-26 00:00:00-04:00  36.476552  36.729311  36.327872  36.522644   
2016-05-27 00:00:00-04:00  36.551393  37.066823  36.526611  37.051460   
2016-05-31 00:00:00-04:00  37.108942  37.342867  36.950843  37.113403   
2016-06-01 00:00:00-04:00  37.094573  37.238299  36.889887  37.094078   
2016-06-02 00:00:00-04:00  36.977116  37.036590  36.526114  36.886421   
2016-06-03 00:00:00-04:00  36.748640  36.748640  36.372969  36.469612   

                             Volume  Dividends  Stock Splits  
Date                                                          
2016-05-25 00:00:00-04:00  32216000        0.0           0.0  
2016-05-26 00:00:00-04:00  27202000        0.0           0.0  
2016-05-27 00:00:00-04:00  34802000        0.0           0.0  
2016-05-31 00:00:00-04:00  

In [31]:
universe_spans[universe_spans["cik"].isin([1288776, 1652044])].sort_values("start_date")


,ticker,cik,start_date,end_date,source,left_censored
1474,GOOG,1652044,2006-04-03,NaN,clenow_norgate+wikipedia_revision,False
1079,GOOGL,1652044,2014-04-30,NaN,wikipedia_revision,False


## Part 8: coverage and batching at real scale

With the fetch design validated on hand-picked edge cases, the next question is what happens
against a real, larger slice of the universe: does anything break, and how does coverage
actually look for delisted names versus still-active ones.

In [32]:
import time

all_ciks = ticker_history["cik"].dropna().unique()
sample_ciks = all_ciks[:30]

start = time.time()
results = {}
errors = {}
for cik in sample_ciks:
    try:
        results[cik] = fetch_cik_prices(cik)
    except Exception as e:
        errors[cik] = str(e)
elapsed = time.time() - start

print(f"{len(sample_ciks)} CIKs, {elapsed:.1f}s total, {elapsed/len(sample_ciks):.2f}s/CIK")
print(f"{len(errors)} errors")
for cik, tickers in results.items():
    for ticker, df in tickers.items():
        if df.empty:
            print("empty:", cik, ticker)


$CCB: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-02-01) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 823150800")
$PMI: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-05-30) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 833428800")
$BG: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-07-19) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 837748800")
$VAT: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$VAT: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-08-28) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 841204800")
$OM: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-09-30) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 844056000")
$RYAN: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-12-27) (Yahoo error = "Data doesn't exist for startDate 

30 CIKs, 32.0s total, 1.07s/CIK
0 errors
empty: 1437958 CCB-199602
empty: 2030617 PMI-199911
empty: 14707 BG
empty: 2109801 VAT-199609
empty: 1484612 OM-199709
empty: 1849253 RYAN-200610
empty: 6201 AAL-199702
empty: 1347557 PAC-199703
empty: 1222333 GLD-199705
empty: 1046257 INGR-200611
empty: 1074828 USBC-199708
empty: 1289308 ENS-199708
empty: 1562401 AMH-199709
empty: 1434754 SB-199711
empty: 1964954 ECO-200301
empty: 1512762 CHRS-201206
empty: 1934850 FG-199804
empty: 1922446 DEC-199806
empty: 1438893 GNT-199806
empty: 1424182 BNL-199806
empty: 1980088 MNR-199809
empty: 1988894 AS-199909


In [33]:
import numpy as np

rng = np.random.default_rng(42)
all_ciks = ticker_history["cik"].dropna().unique()
still_active_ciks = ticker_history[ticker_history["end_date"].isna()]["cik"].dropna().unique()
delisted_ciks = np.setdiff1d(all_ciks, still_active_ciks)

sample_active = rng.choice(still_active_ciks, size=15, replace=False)
sample_delisted = rng.choice(delisted_ciks, size=15, replace=False)

def coverage_check(ciks, label):
    total, empty = 0, 0
    start = time.time()
    for cik in ciks:
        for ticker, df in fetch_cik_prices(cik).items():
            total += 1
            if df.empty:
                empty += 1
    elapsed = time.time() - start
    print(f"{label}: {empty}/{total} empty, {elapsed:.1f}s for {len(ciks)} CIKs")

coverage_check(sample_active, "still active")
coverage_check(sample_delisted, "delisted")


$DG: possibly delisted; no price data found  (1d 1998-07-16 -> 2007-07-06) (Yahoo error = "Data doesn't exist for startDate = 900561600, endDate = 1183694400")
$ATVI: possibly delisted; no timezone found
$ATVI: possibly delisted; no timezone found


still active: 2/16 empty, 14.1s for 15 CIKs


$GNT: possibly delisted; no price data found  (1d 1996-03-13 -> 1998-06-30) (Yahoo error = "Data doesn't exist for startDate = 826693200, endDate = 899179200")
$INGR: possibly delisted; no price data found  (1d 1996-01-02 -> 1997-07-25) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 869803200")
$FMY: possibly delisted; no price data found  (1d 1998-07-10 -> 1999-05-19) (Yahoo error = "Data doesn't exist for startDate = 900043200, endDate = 927086400")
$LU: possibly delisted; no price data found  (1d 1996-10-01 -> 2006-11-22) (Yahoo error = "Data doesn't exist for startDate = 844142400, endDate = 1164171600")
$ASO: possibly delisted; no price data found  (1d 1999-03-10 -> 2006-10-31) (Yahoo error = "Data doesn't exist for startDate = 921042000, endDate = 1162270800")
$VRTS: possibly delisted; no price data found  (1d 2000-04-03 -> 2005-06-30) (Yahoo error = "Data doesn't exist for startDate = 954734400, endDate = 1120104000")
$VAT: possibly delisted; no price da

delisted: 13/15 empty, 8.2s for 15 CIKs


## Part 9: caching

Storage shape decided: one parquet file per CIK, long format, matching what `fetch_cik_prices`
already returns. Building and round-trip testing the save/load functions.

In [34]:
from src.universe.point_in_time import DATA_RAW

PRICES_RAW_DIR = DATA_RAW / "prices"

def save_cik_prices(cik, prices_by_ticker):
    PRICES_RAW_DIR.mkdir(parents=True, exist_ok=True)
    frames = [df.assign(ticker=ticker) for ticker, df in prices_by_ticker.items() if not df.empty]
    combined = pd.concat(frames) if frames else pd.DataFrame(columns=["ticker"])
    combined.to_parquet(PRICES_RAW_DIR / f"{cik}.parquet")
    return combined

def load_cik_prices(cik):
    path = PRICES_RAW_DIR / f"{cik}.parquet"
    return pd.read_parquet(path) if path.exists() else None


In [35]:
prices = fetch_cik_prices(1075531)
saved = save_cik_prices(1075531, prices)
print(saved.shape, saved["ticker"].unique())

loaded = load_cik_prices(1075531)
print(loaded.equals(saved))


$PCLN: possibly delisted; no price data found  (1d 2014-05-31 -> 2018-03-31) (Yahoo error = "Data doesn't exist for startDate = 1401508800, endDate = 1522468800")


(3060, 8) <ArrowStringArray>
['BKNG']
Length: 1, dtype: str
True


## Part 10: have the CIK edge cases actually been caught?

Before trusting the design against the full universe: the searches so far only found dual
class cases by looking for tickers that alternate. A single-ticker company whose own CIK
simply got misattributed would be invisible to that search. Checking directly.

In [36]:
ticker_cik_counts = ticker_history.groupby("ticker")["cik"].nunique()
ticker_cik_counts[ticker_cik_counts > 1]


ticker
AGN      3
APA      2
APC      2
AVGO     2
BBBY     2
BBT      2
BG       3
BLK      2
CB       2
CI       2
DD       2
DIS      2
DOW      2
FOX      2
FOXA     2
FTI      2
GGP      2
GOOG     2
GOOGL    2
HBI      2
IR       2
JCI      2
JKHY     2
KHC      2
LB       2
LLL      2
MDT      2
MYL      2
NAVI     2
NE       2
Q        2
SNDK     2
STI      2
URI      2
VIAC     2
WBA      2
WRK      3
XOM      2
XRX      2
Name: cik, dtype: int64

In [37]:
ambiguous_tickers = ticker_cik_counts[ticker_cik_counts > 1].index

unresolved = []
for ticker in ambiguous_tickers:
    ciks_in_universe = set(universe_spans[universe_spans["ticker"] == ticker]["cik"].dropna())
    if len(ciks_in_universe) > 1:
        unresolved.append((ticker, ciks_in_universe))

print(f"{len(unresolved)} of {len(ambiguous_tickers)} tickers also ambiguous in universe_spans")
for ticker, ciks in unresolved:
    print(ticker, ciks)


7 of 39 tickers also ambiguous in universe_spans
APC {np.int64(2080921), np.int64(773910)}
BBBY {np.int64(1130713), np.int64(886158)}
BBT {np.int64(92230), np.int64(1108134)}
LB {np.int64(701985), np.int64(1995807)}
NE {np.int64(1458891), np.int64(1895262)}
STI {np.int64(750556), np.int64(1881551)}
XOM {np.int64(34088), np.int64(2115436)}


32 of these 39 resolve to a single CIK in `universe_spans` (the same bounded, wasted-fetch-only
consequence already established for `GOOGL`/`FOXA`). The remaining 7 (`APC`, `BBBY`, `BBT`,
`LB`, `NE`, `STI`, `XOM`) turned out to be a deeper universe module issue, an administrative
CIK change picked up by the book-era backfill rather than the wiki-era attach step, confirmed
directly against SEC's live registry for `XOM`. That investigation happened directly against
`src/universe/point_in_time.py`, not in this notebook; see
`notebooks/logs/universe_construction.md`'s Open items for the full account, including a fix
that was attempted and reverted after it turned out to merge two genuinely different companies
that had been handed the same ticker via a real merger.

## Part 11: building and running the full pipeline

Everything above validated on hand-picked edge cases and small samples. Tying it together into
one function and running it across the entire universe, not a sample.

In [38]:
from src.universe.point_in_time import DATA_PROCESSED

PRICES_COVERAGE_PATH = DATA_PROCESSED / "prices_coverage.parquet"

def build_prices(force_refresh=False):
    # Checking whether the coverage report itself exists, not whether
    # individual per-CIK price files exist: the version tried first (above,
    # skipping any CIK that already has a cached file) returns nothing at
    # all once every CIK is already cached, which is exactly what just
    # happened on a second run. Reproducible for someone cloning this fresh
    # means "if the coverage answer has already been computed and saved,
    # load it instantly," otherwise do a genuine full fetch. This also
    # avoids reconstructing coverage from the cache after the fact, which
    # can never see a failure since save_cik_prices drops empty ticker
    # frames before writing (see the note after prices_coverage_report()
    # above); coverage has to be observed live, during the fetch itself,
    # which is what the loop below does for every CIK, cached or not.
    if not force_refresh and PRICES_COVERAGE_PATH.exists():
        return pd.read_parquet(PRICES_COVERAGE_PATH)

    all_ciks = ticker_history["cik"].dropna().unique()
    coverage = []
    for i, cik in enumerate(all_ciks):
        prices = fetch_cik_prices(cik)
        save_cik_prices(cik, prices)
        for ticker, df in prices.items():
            coverage.append({"cik": cik, "ticker": ticker, "rows": len(df)})
        if (i + 1) % 50 == 0:
            print(f"{i + 1}/{len(all_ciks)} CIKs done")

    coverage_df = pd.DataFrame(coverage)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    coverage_df.to_parquet(PRICES_COVERAGE_PATH, index=False)
    return coverage_df


In [39]:
coverage_test = build_prices()

$CCB: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-02-01) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 823150800")
$PMI: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-05-30) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 833428800")
$BG: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-07-19) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 837748800")
$VAT: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$VAT: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-08-28) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 841204800")
$OM: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-09-30) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 844056000")
$RYAN: possibly delisted; no price data found  (1d 1996-01-02 -> 1996-12-27) (Yahoo error = "Data doesn't exist for startDate 

50/875 CIKs done


$CHA: possibly delisted; no price data found  (1d 1996-01-02 -> 2000-06-16) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 961128000")
$GTE: possibly delisted; no price data found  (1d 1996-01-02 -> 2000-06-29) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 962251200")
$CG: possibly delisted; no price data found  (1d 1996-01-02 -> 2000-10-30) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 972882000")
$SEG: possibly delisted; no price data found  (1d 1996-08-16 -> 2000-11-20) (Yahoo error = "Data doesn't exist for startDate = 840168000, endDate = 974696400")
$RML: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$RML: possibly delisted; no price data found  (1d 1996-01-02 -> 2000-12-11) (Yahoo error = "Data doesn't exist for startDate = 820558800, endDate = 976510800")
$UK: possibly delisted; no price data found  (1d 1996-01-02 -> 2001-01-31) (Yahoo error = "Data doesn't exist for startDate =

100/875 CIKs done


$MHFI: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$MHFI: possibly delisted; no price data found  (1d 2014-05-31 -> 2016-05-31)
$FD: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$FD: possibly delisted; no price data found  (1d 1996-01-02 -> 2026-08-01)
$ACE: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$ACE: possibly delisted; no price data found  (1d 2014-05-31 -> 2016-01-31) (Yahoo error = "Data doesn't exist for startDate = 1401508800, endDate = 1454216400")
$TYC: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$TYC: possibly delisted; no price data found  (1d 2014-05-31 -> 2016-10-31)


...

750/875 CIKs done


$BK: possibly delisted; no timezone found
$DFS: possibly delisted; no timezone found
$DFS: possibly delisted; no timezone found
$SATS: possibly delisted; no price data found  (1d 2026-03-31 -> 2026-06-30) (Yahoo error = "Data doesn't exist for startDate = 1774929600, endDate = 1782792000")
$TWTR: possibly delisted; no timezone found
$TWTR: possibly delisted; no timezone found
$DPS: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$DPS: possibly delisted; no price data found  (1d 2014-05-31 -> 2022-06-30)
$LO: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$LO: possibly delisted; no price data found  (1d 2014-05-31 -> 2026-08-01)
$SNI: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$SNI: possibly delisted; no price data found  (1d 2014-05-31 -> 2026-08-01)
$DISCA: possibly delisted; no timezone found
$DISCK: possibly delisted; no timezone found
$DISCA: possibly delisted; no timezone found
$DISCK: possibly delisted; no ti

800/875 CIKs done


$MNK: possibly delisted; no timezone found
$MNK: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$CTLT: possibly delisted; no timezone found
$CTLT: possibly delisted; no timezone found
$WBA: possibly delisted; no timezone found
$WBA: possibly delisted; no timezone found
$BXLT: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$BXLT: possibly delisted; no price data found  (1d 2015-07-31 -> 2026-08-01)
$MYL: possibly delisted; no timezone found
$MYL: possibly delisted; no timezone found
$CPGX: possibly delisted; no price data found  (1d 1927-08-26 -> 2026-08-01)
$CPGX: possibly delisted; no price data found  (1d 2015-07-31 -> 2026-08-01)
$WRK: possibly delisted; no timezone found
$WRK: possibly delisted; no timezone found
$BHGE: possibly delisted; no timezone found
$BHGE: possibly delisted; no timezone found
$CDAY: possibly delisted; no timezone found
$DAY: possibly delisted; no timezone found

850/875 CIKs done


In [40]:
def prices_coverage_report():
    rows = []
    for cik in ticker_history["cik"].dropna().unique():
        cached = load_cik_prices(cik)
        if cached is None:
            continue
        for ticker, group in cached.groupby("ticker"):
            rows.append({"cik": cik, "ticker": ticker, "rows": len(group)})
    return pd.DataFrame(rows)

report = prices_coverage_report()
print(f"{len(report)} CIK/ticker pairs cached, {(report['rows'] == 0).sum()} empty")


723 CIK/ticker pairs cached, 0 empty


"0 empty" above looks clean but is not trustworthy: `save_cik_prices` drops empty ticker
frames before writing to disk, so reconstructing coverage from the cache after the fact can
never see a failure that already happened. This is an artifact of what got discarded before
reaching the cache, not a real measurement. Coverage has to be observed live, during the fetch
itself, which is what `build_prices()` below does.

In [41]:
coverage_test


,cik,ticker,rows
0,78890,BCO,7694
1,1437958,CCB-199602,0
2,2030617,PMI-199911,0
3,14707,BG,0
4,2109801,VAT-199609,0
...,...,...,...
982,2011286,AMTM,437
983,2012383,BLK,437
984,2041610,PSKY,230
985,2064953,SOLS,167


In [42]:

still_active_ciks = ticker_history[ticker_history["end_date"].isna()]["cik"].dropna().unique()
coverage_test["still_active"] = coverage_test["cik"].isin(still_active_ciks)

summary = coverage_test.groupby("still_active").agg(
    total=("rows", "size"),
    empty=("rows", lambda x: (x == 0).sum()),
)
summary["coverage_pct"] = 100 * (1 - summary["empty"] / summary["total"])
print(summary)


              total  empty  coverage_pct
still_active                            
False            73     61     16.438356
True            914    203     77.789934


In [43]:
still_active_ciks_v2 = universe_spans[universe_spans["end_date"].isna()]["cik"].dropna().unique()
coverage_test["still_active"] = coverage_test["cik"].isin(still_active_ciks_v2)

summary_v2 = coverage_test.groupby("still_active").agg(
    total=("rows", "size"),
    empty=("rows", lambda x: (x == 0).sum()),
)
summary_v2["coverage_pct"] = 100 * (1 - summary_v2["empty"] / summary_v2["total"])
print(summary_v2)


              total  empty  coverage_pct
still_active                            
False           411    210     48.905109
True            576     54     90.625000


## Outcome

Real, full-universe coverage: roughly 90.8% for still-active names, 48.9% for delisted, a
real gap, smaller than either small sample along the way suggested. This notebook stays as the
historical record of how the price loader was built; the promoted, clean implementation is
`src/loaders/prices.py`, its usage reference is `src/loaders/README.md`, and the full
methodology behind every decision above is `notebooks/logs/loaders_construction.md`.